# 第11回：化学の知識を特徴量にする

**今日の問い：研究者の知識を、モデルへ渡せる形にするにはどうするか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 化学的仮説を再計算可能な特徴量へ変える
- 追加前後を同じ条件で比較する
- 系列分割と記述子の限界を意識する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- 特徴量設計：既存情報から予測に役立つ表現を作ること
- 記述子：分子構造などを数値で表す量
- アブレーション：要素を足し引きして寄与を調べる比較
- 適用領域：モデルが信頼できる入力範囲

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## TRY：仮説を計算式にする

最適温度78℃からの距離、単位時間あたりの濃度という2つの仮説特徴量を作ります。


In [ ]:
engineered = df.copy()
engineered["temperature_distance"] = abs(engineered["temperature_c"] - 78)
engineered["concentration_per_hour"] = engineered["concentration_m"] / engineered["reaction_time_h"]
engineered[["temperature_c", "temperature_distance", "concentration_per_hour"]].head()


## 同じ検証条件で追加前後を比べる


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

base = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
added = [*base, "temperature_distance", "concentration_per_hour"]
train_idx, valid_idx = train_test_split(engineered.index, test_size=0.25, random_state=42)
for name, columns in {"追加前": base, "追加後": added}.items():
    model = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=150, max_depth=6, random_state=42))
    model.fit(engineered.loc[train_idx, columns], engineered.loc[train_idx, "yield_pct"])
    pred = model.predict(engineered.loc[valid_idx, columns])
    print(name, "MAE:", round(mean_absolute_error(engineered.loc[valid_idx, "yield_pct"], pred), 3))


## CHALLENGE：RDKitでSMILESから記述子を再計算


In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen
    molecule = Chem.MolFromSmiles("CCO")
    print("エタノールの分子量:", round(Descriptors.MolWt(molecule), 3))
    print("エタノールのLogP:", round(Crippen.MolLogP(molecule), 3))
except ImportError:
    print("RDKitは任意です。計算済みのmolecular_weight、logp、tpsa列で本編を進められます。")


## CHANGE

自分の化学的仮説を1つ選び、計算式・予測時点・期待する方向を先に書いてから列を作ります。改善しなくても有益な結果です。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- 特徴量は予測時点で計算できる必要がある
- 追加して悪化する結果も仮説検証として価値がある
- 類似構造への補間と新規骨格への外挿を区別する


In [ ]:
candidate_sets = {"基本": base, "温度距離だけ": [*base, "temperature_distance"], "濃度/時間だけ": [*base, "concentration_per_hour"], "両方": added}
ablation = []
for name, columns in candidate_sets.items():
    estimator = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=150, max_depth=6, random_state=42))
    estimator.fit(engineered.loc[train_idx, columns], engineered.loc[train_idx, "yield_pct"])
    prediction = estimator.predict(engineered.loc[valid_idx, columns])
    ablation.append({"特徴量セット": name, "列数": len(columns), "MAE": mean_absolute_error(engineered.loc[valid_idx, "yield_pct"], prediction)})
display(pd.DataFrame(ablation).sort_values("MAE").round(3))


## よくある誤り

- 意味を説明できない特徴量を大量追加する
- 目的変数由来の値を特徴量にする
- 追加前後で分割やモデルも変える

## SELF-STUDY（任意・30〜60分）

- 自分の仮説特徴量を式・期待方向・反証条件とともに記録する
- RDKitが使える場合は3記述子を再計算し既存列と照合する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. その特徴量はいつ計算できるか
2. 追加効果をどう公平に比較するか
3. 新規骨格で性能が落ちる理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
